In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

Path("releases").parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    "dataset.csv",
    usecols=[
        "Tag",
        "Consumer Claim",
        "Company",
        "Date received",
        "Submitted via",
        "Tags",
        "State"
    ],
    dtype={
        "Tag": "string",
        "Consumer Claim": "string",
        "Company": "string",
        "Submitted via": "string",
        "Tags": "string",
        "State": "string"
    }
)

# affiche les nombres sans notation scientifique avec pandas
pd.set_option("display.float_format", "{:,.2f}".format)

# --------------------------------------------------------------------------
# Normalise les catégories
# --------------------------------------------------------------------------

TAG_MAPPING = {
    # Anciennes catégories → nouvelles catégories
    "Bank account or service":
        "Checking or savings account",

    "Credit reporting":
        "Credit reporting, credit repair services, or other personal consumer reports",

    "Credit card":
        "Credit card or prepaid card",

    "Prepaid card":
        "Credit card or prepaid card",

    "Money transfers":
        "Money transfer, virtual currency, or money service",

    "Virtual currency":
        "Money transfer, virtual currency, or money service",

    "Payday loan":
        "Payday loan, title loan, or personal loan",

    "Consumer Loan":
        "Payday loan, title loan, or personal loan",

    # Catégories déjà dans la nouvelle nomenclature
    "Checking or savings account":
        "Checking or savings account",

    "Debt collection":
        "Debt collection",

    "Credit reporting, credit repair services, or other personal consumer reports":
        "Credit reporting, credit repair services, or other personal consumer reports",

    "Mortgage":
        "Mortgage",

    "Student loan":
        "Student loan",

    "Vehicle loan or lease":
        "Vehicle loan or lease",

    "Credit card or prepaid card":
        "Credit card or prepaid card",

    "Payday loan, title loan, or personal loan":
        "Payday loan, title loan, or personal loan",

    "Money transfer, virtual currency, or money service":
        "Money transfer, virtual currency, or money service",

    "Other financial service":
        "Other financial service",
}

df["Tag"] = df["Tag"].map(TAG_MAPPING)

# --------------------------------------------------------------------------
# Création du dataset
# --------------------------------------------------------------------------

dataset = (
    df
    .loc[df["Tag"] != "Other financial service"]
    .loc[df["Consumer Claim"].notna()]
    .loc[:, [
        "Tag",
        "Consumer Claim"
    ]]
    .drop_duplicates(subset=["Consumer Claim"])
    .copy()
)

# --------------------------------------------------------------------------
# Création du jeu de test/entrainement
# --------------------------------------------------------------------------

X = dataset.drop(columns=["Tag"])
y = dataset["Tag"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,# 20% dans test et 80% dans train
    random_state=42, # seed pour reproduire l'échantillon à l'identique
    stratify=y # Tag possède plusieurs classes, dont certaines beaucoup moins représentées que d'autres
)


In [2]:
dataset.info()

<class 'pandas.DataFrame'>
Index: 366654 entries, 29904 to 912553
Data columns (total 2 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   Tag             366654 non-null  str   
 1   Consumer Claim  366654 non-null  string
dtypes: str(1), string(1)
memory usage: 409.1 MB


In [3]:
dataset.sample(n=10)

,Tag,Consumer Claim
854256,Mortgage,good faith estimate changed 3 times and intere...
692537,Debt collection,I went to see a doctor @ XXXX XXXX XXXX XXXX X...
480264,Mortgage,I applied for an FHA Home loan with First Comm...
474370,Checking or savings account,I signed up for an HSBC premier checking accou...
558534,Credit card or prepaid card,"Money NEt work, First Data, was given my money..."
602187,Credit card or prepaid card,I have been a customer of capital one for 10 y...
684735,Student loan,I contacted Navient ( my servicer ) and reques...
194316,Mortgage,I HAVE APPLIED FOR FINANCIAL ASSISTANCE DUE TO...
391854,Student loan,We are filing a lawsuit in the amount of XXXX ...
291382,Checking or savings account,Contacted CitibankbCustomer Service on XX/XX/X...


# Vectorisation TF-IDF

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path
import time

print("Vectorisation...")
start = time.perf_counter()

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2), # (1, 2) suffit souvent et évite l'explosion de mémoire subie avec (1, 3)
    min_df=10,
    max_df=0.85,
    max_features=100000,
    sublinear_tf=True,
    stop_words=['xxxx', 'xx', 'xxxx xxxx'] # Nettoyer les masques anonymisés CFPB
)
    
# L'entrainement est réalisé uniquement sur le train
vectorizer.fit(X_train["Consumer Claim"])

elapsed = time.perf_counter() - start
print("Terminé en ", elapsed / 60 if elapsed > 60 else elapsed," (min)" if elapsed > 60 else " (s)")

# transforme les données en vecteur
X_train_tfidf = vectorizer.transform(X_train["Consumer Claim"])

Vectorisation...
Terminé en  1.4804859866664628  (min)


# Entrainement du modèle

In [5]:
from sklearn.svm import LinearSVC

print("Entraine du modèle...")
start = time.perf_counter()

svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=5000
)
svm.fit(X_train_tfidf, y_train)

elapsed = time.perf_counter() - start
print("Terminé en ", elapsed / 60 if elapsed > 60 else elapsed," (min)" if elapsed > 60 else " (s)")

Entraine du modèle...
Terminé en  2.2184091616666897  (min)


# Entrainement du modèle avec calibration

In [6]:
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV, FrozenEstimator

print("Entraine du modèle calibré...")
start = time.perf_counter()

# 1. Envelopper le modèle déjà entraîné (svm) dans un FrozenEstimator
frozen_svm = FrozenEstimator(svm)

# 2. Instancier le calibrateur avec l'estimateur figé
calibrated_svm = CalibratedClassifierCV(frozen_svm)

# 3. Ajuster la calibration le jeu de données (X_train_tfidf, y_train)
calibrated_svm.fit(X_train_tfidf, y_train)

elapsed = time.perf_counter() - start
print("Terminé en ", elapsed / 60 if elapsed > 60 else elapsed," (min)" if elapsed > 60 else " (s)")

Entraine du modèle calibré...
Terminé en  4.761434499989264  (s)


# Test

In [10]:
# Définition du seuil de confiance arbitraire pour un premier test (< 70%)
SEUIL_CONFIANCE = 0.95

In [14]:
from pathlib import Path
from sklearn.pipeline import Pipeline
import time
from metrics import (
    calculate_metrics,
    print_metrics,
    print_report,
)

name = "Test ML v1"
method = "TFIDF + LinearSVC + Calibration"
results_path = Path("tests", name + ".pkl")
 
model = Pipeline([
    ("tfidf", vectorizer),
    ("classifier", calibrated_svm)
])

results = []

for idx in X_test.index:
    question = X_test.loc[idx, "Consumer Claim"]
    expected = y_test.loc[idx]

    start = time.perf_counter()

    y_pred = model.predict([question])[0]
    proba = model.predict_proba([question])[0]
    
    elapsed = time.perf_counter() - start

    results.append({
        "Index": idx,
        "Question": question,
        "Réponse": y_pred,
        "Attendue": expected,
        'Confiance': proba.max(),
        "Correct": y_pred == expected,
        "Temps (s)": elapsed
    })

In [18]:
results = pd.DataFrame(results)

results.head(50)

,Index,Question,Réponse,Attendue,Confiance,Correct,Temps (s)
0,895245,not able to get Equifax report are XXXX repot ...,"Credit reporting, credit repair services, or o...","Credit reporting, credit repair services, or o...",1.00,True,0.00
1,550620,My property taxes for year XXXX have not been ...,Mortgage,Mortgage,1.00,True,0.01
2,757469,I ordered a spa cover on XXXX the XXXX from XX...,Credit card or prepaid card,Credit card or prepaid card,0.67,True,0.00
3,719214,My email to RushCard sums it up : I have calle...,Credit card or prepaid card,Credit card or prepaid card,0.87,True,0.01
4,904371,Caliber Home Loans took over the servicing of ...,Mortgage,Mortgage,1.00,True,0.00
5,663227,"Listing account as open, charged off by origin...",Debt collection,Debt collection,0.79,True,0.00
6,735782,"im disputing this account, on the basis that i...",Debt collection,Debt collection,0.95,True,0.01
7,483451,Numerous accounts and inquires on my credit re...,"Credit reporting, credit repair services, or o...","Credit reporting, credit repair services, or o...",1.00,True,0.00
8,837445,We submitted documentation in an attempt to ha...,Mortgage,Mortgage,0.99,True,0.00
9,781784,I accidentally left a check made out to myself...,Checking or savings account,Checking or savings account,1.00,True,0.00


In [16]:

metrics = calculate_metrics(name, method, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
print_report(results)


------------------------
Test ML v1
------------------------
Name : Test ML v1
Method : TFIDF + LinearSVC + Calibration
Samples : 73331
Accuracy : 84.45%
Precision (macro) : 80.19%
Recall (macro) : 72.99%
F1 (macro) : 75.61%
Precision (weighted) : 84.06%
Recall (weighted) : 84.45%
F1 (weighted) : 84.06%
Temps moyen (s) : 0.0035 s
Temps médian (s) : 0.0033 s
Temps P95 (s) : 0.0047 s

------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.82      0.82      0.82      5542
                                                 Credit card or prepaid card       0.81      0.81      0.81      8292
Credit reporting, credit repair services, or other personal consumer reports       0.86      0.89      0.87     22047
                                                             Debt collection       0.82      0.87      0.84     168

# Sauvegarde

In [19]:
import pickle
from metrics import (
    _metrics,
    _report,
)

# sauvegarde les résultats du test
with open("releases/ML_v1.rslt", "wb") as f:
    pickle.dump(results, f)

# sauvegarde le rapport
with open("releases/ML_v1.txt", "w") as f:
    f.write(f"Seuil confiance: {SEUIL_CONFIANCE}" + "\n" + _metrics(metrics) + "\n" + _report(results))

# sauvegarde le modèle
with open("releases/ML_v1.pkl", "wb") as f:
    pickle.dump(model, f)